## What-If Simulation

In [1]:
import pandas as pd, numpy as np, yaml
from pathlib import Path

baseline = pd.read_csv("../data/processed/shipments_simulated.csv")

def kpi_summary(df):
    return {
        "avg_lead_days": df["realized_lead_days"].mean(),
        "on_time_rate": df["delivered_on_time"].mean() * 100,
        "total_cost": df["transport_cost"].sum()
    }

baseline_kpi = kpi_summary(baseline)
baseline_kpi

{'avg_lead_days': np.float64(16.457264957264957),
 'on_time_rate': np.float64(79.7008547008547),
 'total_cost': np.float64(27580.408680000328)}

In [2]:
with open("../config.yaml", "r") as f:
    cfg = yaml.safe_load(f)

scenario_cfg = cfg.copy()
scenario_cfg["lead_time"]["reliability_default"] = 0.85
scenario_cfg["cost"]["fuel_surcharge_pct"] = 0.15
scenario_name = "LowReliability_HighFuel"

In [3]:
lanes = pd.read_csv("../data/processed/lanes_generated.csv")
demand = pd.read_csv("../data/processed/demand_simulated.csv")

In [4]:
def simulate_shipment(retailer_id, sku_id, week, units, lanes_df, reliability_cfg, cost_cfg):
    route = ["supplier","factory","warehouse","retail"]
    total_lead = 0
    total_cost = 0
    for i in range(len(route)-1):
        leg_from, leg_to = route[i], route[i+1]
        lane = lanes_df[(lanes_df.from_type==leg_from)&(lanes_df.to_type==leg_to)].sample(1).iloc[0]
        base_lt = lane["base_lead_days"]
        cost = lane["transport_cost_per_unit"] * units
        # delay logic
        if np.random.rand() > reliability_cfg["lane_reliability"]:
            delay = np.random.randint(1, reliability_cfg["max_delay_days"] + 1)
        else:
            delay = 0
        total_lead += base_lt + delay
        total_cost += cost * (1 + cost_cfg["fuel_surcharge_pct"])
    expected_lead = (len(route)-1) * reliability_cfg["expected_base_days"]
    on_time = total_lead <= expected_lead * 1.1
    return total_lead, on_time, total_cost

In [5]:
reliability_cfg = {
    "lane_reliability": scenario_cfg["lead_time"]["reliability_default"],
    "expected_base_days": int(lanes["base_lead_days"].mean()),
    "max_delay_days": scenario_cfg["lead_time"]["max_delay_days"]
}
cost_cfg = {"fuel_surcharge_pct": scenario_cfg["cost"]["fuel_surcharge_pct"]}

rows = []
for _, r in demand.iterrows():
    lead, ontime, cost = simulate_shipment(r["retailer_id"], r["sku_id"], r["week"], r["demand_units"], lanes, reliability_cfg, cost_cfg)
    rows.append({
        "retailer_id": r["retailer_id"],
        "sku_id": r["sku_id"],
        "week": r["week"],
        "realized_lead_days": lead,
        "delivered_on_time": ontime,
        "transport_cost": cost
    })

scenario_df = pd.DataFrame(rows)
scenario_df.head()

,retailer_id,sku_id,week,realized_lead_days,delivered_on_time,transport_cost
0,RET_1,SKU_1,1,16,True,349.807575
1,RET_1,SKU_1,2,18,False,301.790130
2,RET_1,SKU_1,3,16,True,312.721915
3,RET_1,SKU_1,4,16,True,230.753250
4,RET_1,SKU_1,5,16,True,273.541645


In [6]:
scenario_kpi = kpi_summary(scenario_df)
comparison = pd.DataFrame([
    {"metric":"avg_lead_days","baseline":baseline_kpi["avg_lead_days"],"scenario":scenario_kpi["avg_lead_days"]},
    {"metric":"on_time_rate","baseline":baseline_kpi["on_time_rate"],"scenario":scenario_kpi["on_time_rate"]},
    {"metric":"total_cost","baseline":baseline_kpi["total_cost"],"scenario":scenario_kpi["total_cost"]}
])
comparison["impact_%"] = ((comparison["scenario"] - comparison["baseline"]) / comparison["baseline"]) * 100
comparison.round(2)

,metric,baseline,scenario,impact_%
0,avg_lead_days,16.46,16.87,2.53
1,on_time_rate,79.70,62.18,-21.98
2,total_cost,27580.41,275545.39,899.06


In [7]:
out = Path("../data/processed/whatif_comparison.csv")
comparison.to_csv(out, index=False)
print(f"✅ Saved what-if comparison to {out}")
comparison

✅ Saved what-if comparison to ..\data\processed\whatif_comparison.csv


,metric,baseline,scenario,impact_%
0,avg_lead_days,16.457265,16.872863,2.525318
1,on_time_rate,79.700855,62.179487,-21.983914
2,total_cost,27580.408680,275545.388210,899.062020
